# 🌸 Data Classification Using AI
**DecodeLabs | Industrial Training Kit — Project 2**

---

## 📌 Project Overview
This project demonstrates the complete supervised learning pipeline using the **K-Nearest Neighbors (KNN)** algorithm on the classic **Iris dataset**.

### Pipeline (IPO Framework)
| Stage | Components |
|-------|------------|
| **Input** | Iris Dataset + Feature Scaling |
| **Process** | Train-Test Split + KNN Algorithm |
| **Output** | Confusion Matrix + F1 Score |

### Dataset Info
- **Samples:** 150 (Balanced — 50 per class)
- **Classes:** 3 (Setosa, Versicolor, Virginica)
- **Features:** 4 (Sepal Length, Sepal Width, Petal Length, Petal Width)


## Step 1 — Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn — dataset, preprocessing, model, evaluation
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## Step 2 — Load & Understand the Dataset

In [ ]:
# Load the Iris dataset
iris = load_iris()

# Create a DataFrame for easy exploration
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)

print("📊 Dataset Shape:", df.shape)
print("\n🌸 Class Distribution:")
print(df['species'].value_counts())
print("\n📋 First 5 Rows:")
df.head()

In [ ]:
# Basic statistics
print("📈 Statistical Summary:")
df.describe().round(2)

In [ ]:
# Check for missing values
print("🔍 Missing Values:")
print(df.isnull().sum())
print("\n✅ No missing values — dataset is clean!")

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Pairplot to visualize feature relationships
sns.set_style("whitegrid")
sns.pairplot(df, hue='species', palette='Set2', diag_kind='kde', plot_kws={'alpha': 0.7})
plt.suptitle('Iris Feature Pairplot by Species', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/pairplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Observation: Petal features separate species much better than sepal features.")

In [ ]:
import os
os.makedirs('plots', exist_ok=True)

# Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
corr = df.drop('species', axis=1).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, ax=axes[0], linewidths=0.5)
axes[0].set_title('Feature Correlation Heatmap', fontweight='bold')

# Class distribution bar chart
species_counts = df['species'].value_counts()
colors = ['#2ecc71', '#3498db', '#e74c3c']
axes[1].bar(species_counts.index, species_counts.values, color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('Class Distribution (Balanced)', fontweight='bold')
axes[1].set_xlabel('Species')
axes[1].set_ylabel('Count')
for i, v in enumerate(species_counts.values):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Feature Scaling (The Gatekeeper Rule)

> **Why?** KNN is distance-based. Without scaling, features with larger ranges (e.g., 0–1000) dominate over smaller ones (e.g., 0–1). `StandardScaler` transforms each feature to **mean=0, variance=1**.

In [ ]:
# Separate features (X) and target (y)
X = iris.data
y = iris.target
class_names = iris.target_names

print("Features (X) shape:", X.shape)
print("Target  (y) shape:", y.shape)
print("Classes:", class_names)

In [ ]:
# Train-Test Split BEFORE scaling (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 80% train, 20% test
    random_state=42,      # Reproducibility
    shuffle=True,         # Remove order bias
    stratify=y            # Maintain class balance in both splits
)

print(f"Training set:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set:       {X_test.shape[0]} samples  ({X_test.shape[0]/len(X)*100:.0f}%)")
print("\n✅ Stratified split — class balance maintained!")

In [ ]:
# Apply StandardScaler — fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # only transform on test (no leakage!)

print("Before Scaling — Training set stats:")
print(f"  Mean: {X_train.mean(axis=0).round(2)}")
print(f"  Std:  {X_train.std(axis=0).round(2)}")

print("\nAfter Scaling — Training set stats:")
print(f"  Mean: {X_train_scaled.mean(axis=0).round(2)}")
print(f"  Std:  {X_train_scaled.std(axis=0).round(2)}")

print("\n✅ Data scaled: mean ≈ 0, std ≈ 1")

## Step 5 — Find Optimal K (The Elbow Method)

In [ ]:
# Test K from 1 to 20 and record error rate
k_range = range(1, 21)
error_rates = []
f1_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred_k = knn.predict(X_test_scaled)
    error_rates.append(1 - accuracy_score(y_test, y_pred_k))
    f1_scores.append(f1_score(y_test, y_pred_k, average='weighted'))

# Plot error rate vs K
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, error_rates, 'bo-', markersize=8, linewidth=2)
axes[0].set_xlabel('K Value', fontsize=12)
axes[0].set_ylabel('Error Rate', fontsize=12)
axes[0].set_title('Elbow Method: Error Rate vs K', fontweight='bold')
axes[0].axvline(x=5, color='red', linestyle='--', alpha=0.7, label='K=5 (chosen)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, f1_scores, 'gs-', markersize=8, linewidth=2)
axes[1].set_xlabel('K Value', fontsize=12)
axes[1].set_ylabel('F1 Score (weighted)', fontsize=12)
axes[1].set_title('F1 Score vs K', fontweight='bold')
axes[1].axvline(x=5, color='red', linestyle='--', alpha=0.7, label='K=5 (chosen)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = k_range[np.argmin(error_rates)]
print(f"🏆 Best K by error rate: {best_k}")

## Step 6 — Train the KNN Model

In [ ]:
# Instantiate → Fit → Predict  (The 3-step Scikit-learn workflow)

# INSTANTIATE — Build the frame
model = KNeighborsClassifier(n_neighbors=5)

# FIT — Memorize the training map
model.fit(X_train_scaled, y_train)

# PREDICT — Apply logic to unseen test data
y_pred = model.predict(X_test_scaled)

print("✅ Model trained and predictions generated!")
print(f"\nSample Predictions: {y_pred[:10]}")
print(f"Actual Labels:      {y_test[:10]}")

## Step 7 — Evaluate the Model

> **Why not just accuracy?** On imbalanced datasets, accuracy is a mirage (99% accuracy by always predicting the majority class). We use **F1 Score** (harmonic mean of Precision & Recall) and a **Confusion Matrix** for honest evaluation.

In [ ]:
# Core metrics
accuracy  = accuracy_score(y_test, y_pred)
f1_weighted = f1_score(y_test, y_pred, average='weighted')
f1_macro    = f1_score(y_test, y_pred, average='macro')

print("=" * 45)
print("        📊 MODEL EVALUATION RESULTS        ")
print("=" * 45)
print(f"  Accuracy       : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  F1 Score (weighted): {f1_weighted:.4f}")
print(f"  F1 Score (macro)   : {f1_macro:.4f}")
print("=" * 45)

In [ ]:
# Full classification report
print("📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw count confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, ax=axes[0])
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

# Normalized confusion matrix (percentage)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, ax=axes[1])
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')

plt.suptitle(f'KNN (K=5) — Accuracy: {accuracy*100:.1f}% | F1 Score: {f1_weighted:.3f}',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8 — Decision Boundary Visualization

In [ ]:
# Visualize decision boundary using the 2 most informative features
# (Petal Length & Petal Width — features 2 and 3)

feature_idx = [2, 3]  # petal length, petal width
feature_labels = ['Petal Length (cm)', 'Petal Width (cm)']

X_2d = iris.data[:, feature_idx]
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=0.2, random_state=42, stratify=y
)

scaler_2d = StandardScaler()
X_train_2d_sc = scaler_2d.fit_transform(X_train_2d)
X_test_2d_sc  = scaler_2d.transform(X_test_2d)

knn_2d = KNeighborsClassifier(n_neighbors=5)
knn_2d.fit(X_train_2d_sc, y_train_2d)

# Mesh grid
h = 0.02
x_min, x_max = X_train_2d_sc[:, 0].min() - 1, X_train_2d_sc[:, 0].max() + 1
y_min, y_max = X_train_2d_sc[:, 1].min() - 1, X_train_2d_sc[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot
plt.figure(figsize=(10, 7))
cmap_light = plt.cm.get_cmap('Pastel1', 3)
cmap_bold  = ['#e74c3c', '#3498db', '#2ecc71']

plt.contourf(xx, yy, Z, alpha=0.35, cmap=cmap_light)
plt.contour(xx, yy, Z, colors='gray', linewidths=0.5, alpha=0.5)

for cls, color, name in zip([0, 1, 2], cmap_bold, class_names):
    mask = y_train_2d == cls
    plt.scatter(X_train_2d_sc[mask, 0], X_train_2d_sc[mask, 1],
                c=color, label=f'{name} (train)', edgecolors='k',
                s=60, alpha=0.8)

plt.scatter(X_test_2d_sc[:, 0], X_test_2d_sc[:, 1],
            c=[cmap_bold[i] for i in y_test_2d],
            marker='*', s=200, edgecolors='black', linewidths=1.2,
            label='Test samples', zorder=5)

plt.xlabel(f'Scaled {feature_labels[0]}', fontsize=12)
plt.ylabel(f'Scaled {feature_labels[1]}', fontsize=12)
plt.title('KNN (K=5) Decision Boundary\n(Petal Length vs Petal Width)', fontsize=13, fontweight='bold')
plt.legend(loc='upper left', fontsize=9)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('plots/decision_boundary.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Predict on New Samples

In [ ]:
# Predict on custom new flower measurements
# Format: [sepal_length, sepal_width, petal_length, petal_width]

new_samples = np.array([
    [5.1, 3.5, 1.4, 0.2],  # likely Setosa
    [6.3, 3.3, 4.7, 1.6],  # likely Versicolor
    [7.2, 3.0, 5.8, 1.6],  # likely Virginica
])

# Scale using the SAME scaler fitted on training data
new_samples_scaled = scaler.transform(new_samples)

# Predict
predictions = model.predict(new_samples_scaled)
probabilities = model.predict_proba(new_samples_scaled)

print("🔮 Predictions on New Samples:")
print("-" * 55)
for i, (sample, pred, proba) in enumerate(zip(new_samples, predictions, probabilities)):
    top_prob = proba.max() * 100
    print(f"Sample {i+1}: {sample}")
    print(f"  → Predicted: {class_names[pred].upper()} ({top_prob:.1f}% confidence)")
    print(f"  → Proba: Setosa={proba[0]:.2f}, Versicolor={proba[1]:.2f}, Virginica={proba[2]:.2f}")
    print()

## Step 10 — Final Summary

In [ ]:
print("="*55)
print("        ✅ PROJECT 2 — FINAL SUMMARY           ")
print("="*55)
print(f"  Dataset        : Iris (150 samples, 3 classes, 4 features)")
print(f"  Train/Test Split: 80% / 20% (stratified)")
print(f"  Scaling        : StandardScaler (mean=0, var=1)")
print(f"  Algorithm      : K-Nearest Neighbors")
print(f"  Best K         : 5")
print("-"*55)
print(f"  Accuracy       : {accuracy*100:.2f}%")
print(f"  F1 Score (W)   : {f1_weighted:.4f}")
print(f"  F1 Score (M)   : {f1_macro:.4f}")
print("="*55)
print("\n📁 Plots saved in /plots/ directory")
print("🚀 Pipeline: Load → EDA → Scale → Split → Train → Evaluate")

---

## 📝 Key Learnings

| Concept | What We Learned |
|---|---|
| **Supervised Learning** | Machine derives decision logic from labeled historical data |
| **Feature Scaling** | Mandatory for distance-based algorithms like KNN |
| **Train-Test Split** | Shuffle + stratify to prevent order bias and class imbalance |
| **K Selection** | Elbow method finds optimal K; K=1 overfits, K=100 underfits |
| **Accuracy Mirage** | On imbalanced data, accuracy misleads — always check F1 Score |
| **Confusion Matrix** | Reveals TP, FP, FN, TN — breaks down per-class performance |

---
*DecodeLabs Industrial Training Kit | Batch 2026 | Project 2*